# Méthodologie CNPS : d'où on part, où on en est, où on veut arriver

Ce notebook explore **chaque fichier Parquet produit par le pipeline**, dans l'ordre où ils sont créés,
pour comprendre concrètement ce que contient chacun, pourquoi il existe, et comment il s'articule avec les
autres. L'objectif final du pipeline est de produire des **statistiques de salaires fiables** (moyenne,
médiane, quantiles, Gini...) ventilées par dimension (secteur, sexe, âge, taille d'entreprise...), en
corrigeant le biais introduit par les entreprises qui ne déclarent pas tous les mois.

**Plan du notebook :**
1. Le problème de fond : la non-déclaration
2. `cnps_cleaned.parquet` — la matière première nettoyée
3. `individual_base.parquet` — préparation niveau individu
4. `firm_base.parquet` — agrégation niveau entreprise + panel équilibré
5. `firm_base.parquet` enrichi ANSTAT — secteur CEPICI en plus du secteur CNPS
6. `analytical_base.parquet` — la jointure individus × entreprises
7. Le modèle de déclaration (régression logistique) — `declaration_model.pkl`
8. `firm_base_imputed.parquet` — imputation multiple des salaires manquants
9. `analytical_base.parquet` mis à jour — pondération finale (IPW/AIPW)
10. L'estimation des indicateurs — et le problème actuellement en cours de correction
11. Où on veut arriver : `indicateurs_cnps.xlsx`

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd() / "src"))
from cnps.config import load_config
from cnps.storage import read_parquet, read_pickle, object_exists

cfg = load_config()
mc = cfg.minio
BUCKET = mc.cleaned_bucket
PREFIX = mc.cleaned_prefix

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(40)

print(f"Bucket donnees nettoyees/structurees : {BUCKET}/{PREFIX}")

## 1. Le problème de fond : la non-déclaration

La CNPS reçoit des déclarations mensuelles d'entreprises sur leurs salariés. **Toutes les entreprises ne
déclarent pas tous les mois** (retard administratif, non-conformité temporaire, fermeture...). Si on
calcule un salaire moyen uniquement à partir des entreprises qui déclarent, on obtient une estimation
biaisée : rien ne garantit que les entreprises non-déclarantes ressemblent statistiquement aux
déclarantes (elles peuvent être systématiquement plus petites, plus jeunes, dans certains secteurs...).

C'est un **biais de sélection** classique (Heckman, 1979). Toute la méthodologie du pipeline — modèle de
déclaration, pondération IPW/AIPW, imputation multiple — sert à corriger ce biais. Sans ça, les
statistiques produites seraient de simples moyennes sur un échantillon non représentatif.

## 2. `cnps_cleaned.parquet` — la matière première nettoyée

**Produit par** : `03_nettoyage_donnees.py`. **Contient** : une ligne par déclaration individuelle
(un salarié, un mois), tous mois confondus. C'est la concaténation de tous les fichiers Excel mensuels
bruts, avec des variables dérivées calculées (âges, anciennetés, classes) et les valeurs de salaire
extrêmes winsorisées.

Aucune agrégation, aucune jointure : c'est la donnée consolidée telle qu'elle vient de la CNPS, juste
apurée.

In [ ]:
df_cleaned = read_parquet(mc, BUCKET, f"{PREFIX}cnps_cleaned.parquet")
print(f"{df_cleaned.height:,} lignes, {df_cleaned.width} colonnes")
df_cleaned.head(5)

### Dictionnaire des colonnes de `cnps_cleaned.parquet`

| Colonne | Description |
|---|---|
| `ID_INDIV` | Identifiant unique du salarié (assigné par la CNPS) |
| `ID_EMPLOYEUR` | Identifiant unique de l'entreprise employeuse |
| `RAISON_SOCIALE` | Nom de l'entreprise (tel que déclaré, non normalisé) |
| `SEXE` | Sexe du salarié (`M`/`F`) |
| `DATE_NAISSANCE` | Date de naissance du salarié |
| `AGE_EMPLOYE` | Âge calculé à la date du jour (dérivé de `DATE_NAISSANCE`) |
| `CL_AGE_EMPLOYE` | Classe d'âge (`Moins de 25 ans`, `25-34 ans`, `35-49 ans`, `50 ans et plus`) |
| `SITUATION_MATRIMONIALE` | Statut marital du salarié |
| `NIVEAU_ETUDE` | Niveau d'étude du salarié |
| `PROFESSION` | Profession déclarée |
| `STATUT_TRAVAILLEUR` | Statut du travailleur (ex: cadre, employé...) |
| `TYPE_SALARIE` | Type de contrat (le nettoyage exclut `H`=horaire et `J`=journalier, cf. `cleaning.exclude_employee_types`) |
| `DATE_EMBAUCHE` | Date d'embauche dans l'entreprise actuelle |
| `ANCIENNETE_ENTREPRISE` | Ancienneté en années (dérivé de `DATE_EMBAUCHE`) |
| `CL_ANCIENNETE_ENTREPRISE` | Classe d'ancienneté (`Moins de 2 ans`, `2-4 ans`, `5-9 ans`, `10 ans et plus`) |
| `SALAIRE_BRUT` | Salaire brut déclaré sur la période |
| `DUREE_TRAVAILLEE` | Durée travaillée en mois sur la période (max 12, cf. `cleaning.max_duration`) |
| `SALAIRE_BRUT_MENS` | **Salaire mensualisé** = `SALAIRE_BRUT / max(DUREE_TRAVAILLEE, 1) * 12` — variable d'intérêt principale |
| `SECTEUR_ACTIVITE` | Secteur d'activité de l'entreprise, **nomenclature CNPS native** (36 valeurs distinctes) |
| `CATEGORIE_ENTREPRISE` | Catégorie de l'entreprise |
| `EFFECTIF_SALARIES` | Effectif salarié déclaré par l'entreprise |
| `COMMUNE` / `LOCALITE` | Localisation de l'entreprise |
| `LIBELLE_AGENCE` | Agence CNPS de rattachement |
| `DATE_IMMATRICULATION` | Date d'immatriculation du salarié à la CNPS |
| `ANCIENNETE_IMMAT` | Ancienneté d'immatriculation (dérivé) |
| `CL_ANCIENNETE_IMMAT` | Classe d'ancienneté d'immatriculation |
| `DATE_IMMAT_EMPLOYEUR` | Date d'immatriculation de l'entreprise |
| `AGE_ENTREPRISE_IMMAT` | Âge de l'entreprise depuis son immatriculation (dérivé) |
| `CL_AGE_ENTREPRISE` | Classe d'âge de l'entreprise |
| `DATE_DEBUT_ACT_EMPLOY` | Date de début d'activité de l'employeur |
| `MOIS` / `ANNEE` / `PERIOD` / `PERIODE` | Découpage temporel (`PERIOD` = `MM_YYYY`, clé de period utilisée dans tout le pipeline) |
| `TRIMESTRE` / `SEMESTRE` | Dérivés de `MOIS` |
| `TAG` | Champ technique interne au fichier source |

**Note :** certaines colonnes existent dans le fichier Excel brut mais ne sont pas listées dans les
listes de traitement explicite du code (`_ID_COLS`, `_NUMERIC_COLS`, `_DATE_COLS` de
`02_harmonisation_types.py`) — elles passent telles quelles (ex: `RAISON_SOCIALE`, `SECTEUR_ACTIVITE`,
`SITUATION_MATRIMONIALE`). C'est normal : seules les colonnes qui ont besoin d'un cast de type explicite
sont listées.

In [ ]:
# Valeurs manquantes par colonne
df_cleaned.null_count().transpose(include_header=True, header_name="colonne", column_names=["n_null"]).with_columns(
    (pl.col("n_null") / df_cleaned.height * 100).round(2).alias("pct_null")
).sort("pct_null", descending=True).head(15)

## 3. `individual_base.parquet` — préparation niveau individu

**Produit par** : `04_base_individus.py`. C'est une **quasi-copie** de `cnps_cleaned.parquet` (même
nombre de lignes) : aucune agrégation, aucune jointure. Seuls ajouts :
- `OBS_ID` : identifiant unique d'observation (`ID_INDIV` + `ID_EMPLOYEUR` + `PERIOD` concaténés)
- `W_INDIV` : poids individuel initial, fixé à 1.0 (sera combiné plus tard avec le poids entreprise)
- `S_IJT` : indicateur d'observation (1 = ce salarié est bien observé sur cette période)

Ce fichier existe comme **étape de préparation légère** avant la jointure avec les entreprises — pas de
transformation lourde ici.

In [ ]:
df_indiv = read_parquet(mc, BUCKET, f"{PREFIX}individual_base.parquet")
print(f"{df_indiv.height:,} lignes, {df_indiv.width} colonnes")
print("Colonnes ajoutees par rapport a cnps_cleaned.parquet :", set(df_indiv.columns) - set(df_cleaned.columns))
df_indiv.select("OBS_ID", "W_INDIV", "S_IJT").head(5)

## 4. `firm_base.parquet` — agrégation niveau entreprise + panel équilibré

**Produit par** : `05_base_entreprises.py`. Ici la transformation est réelle : on passe de "un salarié,
un mois" à **"une entreprise, un mois"**. Toutes les lignes individus d'une même entreprise sur un même
mois sont résumées en une seule ligne (moyenne, somme, pourcentage...).

### Étape A — Agrégation (`group_by(["ID_EMPLOYEUR", "PERIOD", ...])`)

| Colonne produite | Calcul |
|---|---|
| `EFFECTIF_OBSERVE` | Nombre de salariés observés dans cette entreprise ce mois-ci |
| `SALAIRE_MOYEN` | Moyenne de `SALAIRE_BRUT_MENS` sur les salariés de l'entreprise |
| `SALAIRE_MEDIAN` | Médiane du salaire mensuel |
| `MASSE_SALARIALE` | Somme des salaires (masse salariale totale du mois) |
| `SALAIRE_SD` | Écart-type des salaires |
| `PCT_FEMMES` | % de femmes parmi les salariés observés |
| `AGE_MOYEN` | Âge moyen des salariés |
| `ANCIENNETE_MOYENNE` | Ancienneté moyenne des salariés |
| `SECTEUR_ACTIVITE`, `COMMUNE`, `CLASSE_EFFECTIF`, `CLASSE_EFFECTIF_REDUITE`, `AGE_ENTREPRISE_IMMAT`, `CL_AGE_ENTREPRISE` | Attributs entreprise, reportés tels quels (première valeur rencontrée — ce sont des attributs d'entreprise, identiques pour tous ses salariés) |

**Important :** `SEXE`, `AGE_EMPLOYE`, `PROFESSION` (attributs *individuels*) n'existent plus après cette
agrégation — seul leur résumé statistique existe (`PCT_FEMMES`, `AGE_MOYEN`). C'est volontaire : au niveau
entreprise, "le sexe" n'a pas de sens.

### Étape B — Panel équilibré (le point clé de toute la méthodologie)

On construit le **produit cartésien** de toutes les entreprises connues × tous les mois de l'historique
(`all_firms.join(all_periods, how="cross")`), puis on rattache les vraies données là où elles existent. Le
résultat matérialise explicitement les "trous" : chaque entreprise a désormais une ligne pour **chaque**
mois, qu'elle ait déclaré ou non.

| Colonne | Description |
|---|---|
| `D_JT` | **Indicateur de déclaration** : 1 si l'entreprise `j` a déclaré au mois `t`, 0 sinon (0 quand la ligne vient du produit cartésien sans correspondance réelle) |
| `W_JT` | Poids entreprise (initialisé à 1.0, sera recalculé à l'étape 07) |
| `LOG_SALAIRE_MOYEN` | `log(SALAIRE_MOYEN)` — variable cible du modèle d'imputation (étape 08) |
| `LAG_D_JT`, `LAG_SALAIRE_MOYEN`, `LAG_EFFECTIF_OBSERVE` | Valeurs du mois précédent, décalées par entreprise (`.shift(1).over("ID_EMPLOYEUR")`) — utilisées comme prédicteurs : "l'entreprise a-t-elle déclaré le mois dernier ?" |
| `TAUX_DECLARATION_PASSE` | Moyenne cumulée de `D_JT` sur les mois précédents (historique de fiabilité de l'entreprise) |

In [ ]:
df_firm = read_parquet(mc, BUCKET, f"{PREFIX}firm_base.parquet")
print(f"{df_firm.height:,} lignes, {df_firm.width} colonnes")
print()
print("Repartition D_JT (declarant vs non-declarant) :")
print(df_firm["D_JT"].value_counts().sort("D_JT"))

In [ ]:
df_firm.select(
    "ID_EMPLOYEUR", "PERIOD", "D_JT", "EFFECTIF_OBSERVE", "SALAIRE_MOYEN",
    "SECTEUR_ACTIVITE", "CLASSE_EFFECTIF_REDUITE", "LAG_D_JT", "TAUX_DECLARATION_PASSE",
).head(10)

## 5. Jointure ANSTAT — secteur CEPICI en plus du secteur CNPS

**Produit par** : `05_1_jointure_anstat.py`, une étape intercalée entre 05 et 06. `SECTEUR_ACTIVITE`
existe **déjà nativement** dans les données CNPS (vu section 2) — cette étape n'est donc pas la seule
source de secteur, elle ajoute une **deuxième nomenclature** (CEPICI/ANSTAT), potentiellement plus fine
ou différente, en plus d'attributs qu'ANSTAT a et que CNPS n'a pas (forme juridique, RCCM, DFE).

**Méthode** : jointure sur `RAISON_SOCIALE` normalisée (majuscules, ponctuation retirée) — aucun
identifiant commun n'existe entre les deux sources. ~96.5% des entreprises CNPS trouvent une
correspondance.

| Colonne ajoutée | Origine |
|---|---|
| `SECTEUR_ACTIVITE_ANSTAT` | Secteur d'activité selon la nomenclature CEPICI (différente de `SECTEUR_ACTIVITE` CNPS) |
| `FORME_JURIDIQUE_ANSTAT` | Forme juridique de l'entreprise (SARL, SA...) |
| `NUMERO_RCCM` | Numéro de Registre du Commerce et du Crédit Mobilier |
| `NUMERO_DFE` | Numéro de Déclaration Fiscale d'Existence |

In [ ]:
cols_anstat = [c for c in ["SECTEUR_ACTIVITE_ANSTAT", "FORME_JURIDIQUE_ANSTAT", "NUMERO_RCCM", "NUMERO_DFE"] if c in df_firm.columns]
print("Colonnes ANSTAT presentes dans firm_base.parquet :", cols_anstat)
if cols_anstat:
    pct_matched = df_firm.filter(pl.col(cols_anstat[0]).is_not_null()).height / df_firm.height * 100
    print(f"Pct de lignes avec correspondance ANSTAT : {pct_matched:.1f}%")
    df_firm.select(["ID_EMPLOYEUR", "SECTEUR_ACTIVITE"] + cols_anstat).filter(pl.col(cols_anstat[0]).is_not_null()).head(5)

## 6. `analytical_base.parquet` — la jointure individus × entreprises

**Produit par** : `06_base_analytique.py`. C'est la jointure centrale du pipeline : chaque ligne
individu (`individual_base.parquet`) va chercher, via `["ID_EMPLOYEUR", "PERIOD"]`, la ligne entreprise
correspondante (`firm_base.parquet`) et récupère ses attributs (`D_JT`, `W_JT`, secteur, taille...).

C'est une jointure **plusieurs-vers-un** classique (`indiv.join(firm_subset, on=["ID_EMPLOYEUR", "PERIOD"], how="left")`) :
plusieurs salariés d'une même entreprise-mois pointent vers la même ligne entreprise, donc récupèrent
tous les mêmes attributs entreprise, dupliqués sur chacune de leurs lignes.

**Résultat : `analytical_base.parquet` est le SEUL fichier qui réunit toutes les dimensions ensemble** —
attributs individu (`SEXE`, `AGE_EMPLOYE`, `PROFESSION`, `SALAIRE_BRUT_MENS`) ET attributs entreprise
(`SECTEUR_ACTIVITE`, `CLASSE_EFFECTIF_REDUITE`, `D_JT`) sur la même ligne. C'est la base qui **devrait**
toujours servir de source à l'estimation finale des indicateurs (section 10).

In [ ]:
df_analytical = read_parquet(mc, BUCKET, f"{PREFIX}analytical_base.parquet")
print(f"{df_analytical.height:,} lignes, {df_analytical.width} colonnes")
print()
print("Colonnes individu presentes :", [c for c in ["SEXE","AGE_EMPLOYE","PROFESSION","SALAIRE_BRUT_MENS"] if c in df_analytical.columns])
print("Colonnes entreprise presentes :", [c for c in ["SECTEUR_ACTIVITE","CLASSE_EFFECTIF_REDUITE","D_JT","W_JT"] if c in df_analytical.columns])

In [ ]:
df_analytical.select(
    "ID_INDIV", "ID_EMPLOYEUR", "PERIOD", "SEXE", "AGE_EMPLOYE",
    "SALAIRE_BRUT_MENS", "SECTEUR_ACTIVITE", "D_JT", "W_FINAL",
).head(10)

## 7. Le modèle de déclaration — `declaration_model.pkl`

**Produit par** : `07_modele_declaration.py`, entraîné sur **`firm_base.parquet`** (pas
`analytical_base.parquet` !). Question posée : *"quelle est la probabilité qu'une entreprise déclare ce
mois-ci ?"* — c'est une question au niveau entreprise-mois, donc `firm_base.parquet` (une ligne =
entreprise-mois) est la bonne base pour l'entraîner.

**Modèle** : régression logistique — `P(D_JT=1) ~ SECTEUR_ACTIVITE + CLASSE_EFFECTIF_REDUITE + CL_AGE_ENTREPRISE + LAG_D_JT + TAUX_DECLARATION_PASSE`

**Sorties, réécrites dans `firm_base.parquet`** :
- `P_HAT_JT` : score de propension estimé (probabilité prédite de déclarer)
- `W_JT` : poids IPW stabilisé = `P(D=1) / P_HAT_JT`, tronqué aux percentiles configurés (`ipw_trim_lower`/`ipw_trim_upper`, 1%/99% par défaut)

Le modèle lui-même (objet scikit-learn + métriques) est sauvegardé dans `declaration_model.pkl`.

In [ ]:
model_object = f"{mc.models_prefix}declaration_model.pkl"
if object_exists(mc, mc.models_bucket, model_object):
    saved = read_pickle(mc, mc.models_bucket, model_object)
    print("AUC du modele :", saved.get("auc"))
    print("Features utilisees :", saved.get("features"))
else:
    print("Modele pas encore genere sur cette execution.")

In [ ]:
df_firm_weighted = read_parquet(mc, BUCKET, f"{PREFIX}firm_base.parquet")
if "P_HAT_JT" in df_firm_weighted.columns:
    print("Poids IPW (W_JT) :")
    print(df_firm_weighted.select("W_JT").describe())
    print()
    print("Score de propension (P_HAT_JT) :")
    print(df_firm_weighted.select("P_HAT_JT").describe())

## 8. `firm_base_imputed.parquet` — imputation multiple des salaires manquants

**Produit par** : `08_imputation_salaires.py`, toujours au niveau **entreprise**, à partir de
`firm_base.parquet` (qui contient maintenant `D_JT`, `W_JT`, `P_HAT_JT` depuis l'étape 07).

**Problème à résoudre** : pour les entreprises non-déclarantes (`D_JT=0`), `SALAIRE_MOYEN` est inconnu —
on ne peut pas juste ignorer ces lignes (biais de sélection, section 1), il faut estimer ce qu'aurait été
leur salaire moyen.

**Méthode (imputation multiple, Rubin 1987)** :
1. Entraîne `log(SALAIRE_MOYEN) ~ SECTEUR_ACTIVITE + CLASSE_EFFECTIF_REDUITE + CL_AGE_ENTREPRISE + LAG_SALAIRE_MOYEN + LAG_EFFECTIF_OBSERVE`
   **uniquement sur les entreprises déclarantes**
2. Pour chaque entreprise non-déclarante, génère **M=5 versions différentes** du salaire imputé, chacune
   avec un bruit aléatoire différent (`e^(m) ~ N(0, sigma_hat^2)`) — pas une seule valeur "devinée", mais 5
   versions plausibles qui représentent l'incertitude de l'estimation
3. Empile le tout : `firm_base_imputed.parquet` = (déclarantes × 5 copies identiques) + (non-déclarantes ×
   5 versions imputées différentes), avec une colonne `IMPUTATION_ID` (1 à 5)

**Résultat : ce fichier est ~5x plus long que `firm_base.parquet`, et reste au niveau entreprise-mois —
jamais au niveau individu.** C'est le point de départ du bug actuellement en cours de correction
(section 10).

In [ ]:
imputed_object = f"{PREFIX}firm_base_imputed.parquet"
if object_exists(mc, BUCKET, imputed_object):
    df_imputed = read_parquet(mc, BUCKET, imputed_object)
    print(f"{df_imputed.height:,} lignes, {df_imputed.width} colonnes")
    print(f"(firm_base.parquet avait {df_firm.height:,} lignes -> ratio {df_imputed.height / df_firm.height:.1f}x)")
    print()
    print("Colonnes individu presentes dans ce fichier (attendu : AUCUNE) :")
    print([c for c in ["SEXE","AGE_EMPLOYE","PROFESSION"] if c in df_imputed.columns] or "-> confirme : aucune colonne individu ici")
    print()
    print(df_imputed["IMPUTATION_ID"].value_counts().sort("IMPUTATION_ID"))
else:
    print("firm_base_imputed.parquet n'existe pas encore sur MinIO.")

## 9. Pondération finale — `analytical_base.parquet` mis à jour

**Produit par** : `09_ponderation_finale.py`. On revient au niveau **individu** : `analytical_base.parquet`
(créé section 6) est relu et reçoit sa colonne définitive de poids, `W_FINAL`.

**Deux méthodes possibles** (configurées via `modeling.estimation_method`) :
- **IPW simple** : `W_FINAL = W_JT * W_INDIV`
- **AIPW** (doublement robuste, méthode par défaut) : combine le modèle de propension (étape 07) **et**
  le modèle de résultat (imputation, étape 08) dans une seule formule. Reste valide même si l'un des deux
  modèles est mal spécifié (Bang & Robins, 2005) — c'est la garantie statistique de robustesse de cette
  méthode.

Les poids sont ensuite normalisés par période (`W_FINAL / moyenne(W_FINAL) sur cette PERIOD`).

In [ ]:
df_analytical_weighted = read_parquet(mc, BUCKET, f"{PREFIX}analytical_base.parquet")
print("Poids finaux (W_FINAL) :")
print(df_analytical_weighted.select("W_FINAL").describe())

## 10. Estimation des indicateurs — où on en est actuellement

**Objectif de cette étape** (`10_estimation_indicateurs.py`) : calculer, pour chaque dimension
d'analyse (secteur, sexe, âge, taille d'entreprise, CSP, profession, commune — définies dans
`config/dimensions.yaml`), les statistiques configurées (moyenne, médiane, quantiles, Gini...) pondérées
par `W_FINAL`.

**Le bug en cours de diagnostic** : le code vérifie juste "est-ce que `firm_base_imputed.parquet`
existe ?" (section 8) et si oui — ce qui est **toujours** le cas, puisque l'étape 08 le crée
systématiquement — il utilise ce fichier **à la place de** `analytical_base.parquet` (section 9) pour
calculer les indicateurs. Or on a vu section 8 que `firm_base_imputed.parquet` est au niveau
**entreprise**, sans `SEXE`, `AGE_EMPLOYE`, `PROFESSION`, `CSP`...

**Conséquence concrète** : les indicateurs par sexe, âge, ancienneté, CSP, profession ne peuvent jamais
être calculés (colonnes absentes), alors que ces mêmes colonnes existent bel et bien dans
`analytical_base.parquet` (section 6/9) — le seul fichier qui réunit dimensions individu et poids
finaux ensemble.

**La correction à faire** (pas encore appliquée) : l'estimation doit toujours utiliser
`analytical_base.parquet`. La question qui reste à trancher est *comment* continuer à appliquer les
règles de combinaison de Rubin (propager l'incertitude des 5 imputations dans les intervalles de
confiance) sur cette base niveau-individu plutôt que sur `firm_base_imputed.parquet`.

In [ ]:
from cnps.config import PipelineConfig  # noqa: F401  (juste pour verifier l'import)

print("Dimensions configurees dans dimensions.yaml :")
for dim in cfg.dimensions:
    if dim.enabled:
        cols_ok = [c for c in dim.group_by if c in df_analytical_weighted.columns]
        cols_ko = [c for c in dim.group_by if c not in df_analytical_weighted.columns]
        statut = "OK (dans analytical_base)" if not cols_ko else f"MANQUANT : {cols_ko}"
        print(f"  - {dim.label:40s} group_by={dim.group_by}  [{statut}]")

In [ ]:
# Preuve directe du probleme : ces colonnes existent dans analytical_base.parquet...
cols_test = ["SEXE", "CL_AGE_EMPLOYE", "CL_ANCIENNETE_ENTREPRISE", "SECTEUR_ACTIVITE"]
print("Dans analytical_base.parquet (utilise section 9) :")
print([c for c in cols_test if c in df_analytical_weighted.columns])

print()
print("Dans firm_base_imputed.parquet (utilise a tort par l'etape 10 actuellement) :")
if object_exists(mc, BUCKET, imputed_object):
    print([c for c in cols_test if c in df_imputed.columns] or "AUCUNE -> c'est la cause du bug")

## 11. Où on veut arriver : `indicateurs_cnps.xlsx`

**Produit par** : `12_export_excel.py`, à partir du résultat de l'étape 10. Le fichier final contient,
pour chaque dimension activée (secteur, sexe, âge, taille d'entreprise, etc.) et chaque groupe (ex:
"Femmes", "35-49 ans", "Commerce"...) :

- L'effectif non pondéré et pondéré (`n_obs`, `n_weighted`)
- Le salaire moyen, médian, les déciles/quartiles pondérés
- L'écart-type, le minimum, le maximum
- Le coefficient de Gini (inégalité de la distribution)
- Pour chaque statistique : l'intervalle de confiance à 95% (combiné via les règles de Rubin sur les 5
  imputations) et la fraction d'information manquante (FMI)

Les cellules dont l'effectif pondéré est inférieur à 30 (`estimation.min_cell_size`) sont supprimées de
la diffusion, conformément aux standards de statistique publique (INSEE, Eurostat) — évite de publier des
statistiques peu fiables ou identifiantes sur de trop petits groupes.

**Une fois le bug de la section 10 corrigé**, ce fichier final contiendra des ventilations complètes par
sexe, âge, ancienneté, secteur (CNPS et ANSTAT), taille d'entreprise — pas seulement les dimensions
disponibles au niveau entreprise.

In [ ]:
output_object = f"{mc.output_prefix}indicateurs_cnps.xlsx"
if object_exists(mc, mc.output_bucket, output_object):
    print(f"Export deja genere : {mc.output_bucket}/{output_object}")
else:
    print("Export pas encore genere sur cette execution (bloque par le bug de l'etape 10).")